# Google Places API (New): Exploring Urban Places with Python

**RC15 Resources · Mapping and urban data**

This exercise follows the idea of the original *Google Maps Services* notebook from RC15 Skills 2025–26: use Python to find a place, examine its structured data and retrieve photographs. The original uses Google Places API (Legacy). This **new notebook** uses **Places API (New)** and keeps the legacy notebook unchanged for comparison.

**What you will learn:** read JSON, make a Places Text Search request, retrieve Place Details, interpret coordinates and attributes, and examine photo attribution. The first exercise works **offline**, without an API key. The remaining requests are optional and may incur costs.

**Before using live requests:** read [API_SETUP.md](API_SETUP.md), create a Google Cloud project, enable **Places API (New)**, review billing and restrict your API key. Never save keys in a notebook or commit them to GitHub.

**Status:** Code and response formats checked against official Google documentation (October 2026); **not tested against a billed Google Cloud account**. All notebook outputs have been cleared.

## 1 · Read place data as JSON, without an API key

An API returns structured information, commonly formatted as **JSON** (JavaScript Object Notation). JSON objects contain keys and values, while arrays contain ordered collections of objects. In Python, JSON objects become dictionaries and arrays become lists.

We begin with a small **fictional** response using the structure of **Places API (New)**. This makes the data easier to understand before we make a billable request.

In [ ]:
example_response = {
    "places": [
        {
            "id": "EXAMPLE_PLACE_ID",
            "displayName": {"text": "Example Urban Library"},
            "formattedAddress": "Example Street, London",
            "location": {"latitude": 51.52, "longitude": -0.12},
        }
    ]
}

place = example_response["places"][0]
print("Name:", place["displayName"]["text"])
print("Address:", place["formattedAddress"])
print("Latitude:", place["location"]["latitude"])
print("Longitude:", place["location"]["longitude"])

### Compare the archived response

The folder also contains [`assets/sample.json`](assets/sample.json), an actual historical **Places API (Legacy)** response from the original exercise. It is retained for comparison, not as a sample of the current API.

Notice that the old JSON has a top-level `result` object, uses `geometry.location.lat/lng`, and stores photo identifiers as `photo_reference`. The new Text Search returns a top-level `places` array with `location.latitude/longitude`; it uses **photo resource names** rather than legacy photo references.

This cell reads the local file and reports only a few structural facts.

In [ ]:
import json
from pathlib import Path

sample_path = Path("assets/sample.json")
if sample_path.exists():
    with sample_path.open(encoding="utf-8") as f:
        legacy_data = json.load(f)
    legacy_place = legacy_data.get("result", {})
    print("Legacy response keys:", list(legacy_data))
    print("Place name:", legacy_place.get("name", "(missing)"))
    print("Photo references:", len(legacy_place.get("photos", [])))
else:
    print("Not found:", sample_path, "— open the notebook from its tutorial folder.")

## 2 · Prepare for an optional live request

Unlike the offline exercise, calls to Google's service require a **Google Cloud project**, an enabled **Places API (New)** and a valid API key. Billing may apply. The code deliberately does not make any live request until you set `RUN_LIVE_REQUESTS = True`.

Set the `GOOGLE_MAPS_API_KEY` environment variable **outside** the notebook, following [API_SETUP.md](API_SETUP.md). In the notebook we only check whether it is present; we never print its value.

We use Python's built-in `urllib` and `json` modules, so no separate Google client library is required.

In [ ]:
import os
from urllib.error import HTTPError, URLError
from urllib.parse import quote, urlencode
from urllib.request import Request, urlopen

RUN_LIVE_REQUESTS = False  # Change deliberately after checking billing and restrictions.
API_ROOT = "https://places.googleapis.com/v1"
api_key_available = bool(os.getenv("GOOGLE_MAPS_API_KEY"))
print("API key available:", api_key_available)
print("Live requests enabled:", RUN_LIVE_REQUESTS)

### A small HTTP helper

A **request** is a message sent to a web service; a **response** contains its answer. Places API (New) expects an API key in the `X-Goog-Api-Key` header. It also requires a **field mask** for Text Search and Place Details: a list of the precise fields to return.

Field masks limit what we receive and can affect pricing. Avoid requesting every possible field using `*`. The helper below reports HTTP errors without printing the API key.

In [ ]:
def places_json_request(endpoint, field_mask, payload=None):
    """Make a Places API (New) request and return its JSON response.

    endpoint: a full Google Places API endpoint
    field_mask: comma-separated response fields
    payload: None for GET, or a dict for POST
    """
    api_key = os.getenv("GOOGLE_MAPS_API_KEY")
    if not api_key:
        raise RuntimeError("Set GOOGLE_MAPS_API_KEY outside this notebook first.")

    data = None if payload is None else json.dumps(payload).encode("utf-8")
    headers = {
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": field_mask,
        "Content-Type": "application/json",
    }
    req = Request(endpoint, data=data, headers=headers,
                  method="POST" if payload is not None else "GET")

    try:
        with urlopen(req, timeout=20) as response:
            return json.load(response)
    except HTTPError as exc:
        # Do not print request headers, API keys or complete signed URLs.
        raise RuntimeError(f"Places API returned HTTP {exc.code}. Check the API, "
                           "key restrictions, field mask and billing.") from None
    except URLError:
        raise RuntimeError("Connection failed. Check your network connection.") from None

## 3 · Find a place with Text Search (New)

A **Text Search** turns a description or name into candidate places. For an urban study, try a public library, market or community space. Start with a single query and at most three results; the output can vary over time.

**Text Search endpoint:** `POST https://places.googleapis.com/v1/places:searchText`. The example requests a place ID, display name, address and geographical coordinates.

In [ ]:
query = "public libraries in Hackney, London"
search_results = None

if RUN_LIVE_REQUESTS:
    search_results = places_json_request(
        f"{API_ROOT}/places:searchText",
        "places.id,places.displayName,places.formattedAddress,places.location",
        payload={"textQuery": query, "pageSize": 3},
    )
    for index, item in enumerate(search_results.get("places", [])):
        print(index, item.get("displayName", {}).get("text"),
              "—", item.get("formattedAddress", ""))
else:
    print("Offline mode: no Text Search request sent.")

### Study the returned attributes

The resulting `places` array contains dictionaries. You can use a list comprehension to extract selected values and examine how Google represents a location.

These are **provider-supplied place records**, not a complete or unbiased census of all urban activities. Availability, naming and category coverage differ between places and over time.

In [ ]:
if search_results:
    rows = [
        {
            "name": p.get("displayName", {}).get("text"),
            "latitude": p.get("location", {}).get("latitude"),
            "longitude": p.get("location", {}).get("longitude"),
            "place_id": p.get("id"),
        }
        for p in search_results.get("places", [])
    ]
    for row in rows:
        print(row)
else:
    print("Run the optional Text Search first to extract live results.")

## 4 · Inspect one place with Place Details (New)

Text Search returns a short list of places. **Place Details** uses a selected **place ID** to request information about an individual location.

**Place Details endpoint:** `GET https://places.googleapis.com/v1/places/{PLACE_ID}`.

For this exercise we ask for its name, address, coordinates and the **names and attribution metadata** of available photographs. Requesting different fields can affect cost. We are deliberately not retrieving ratings, reviews or opening hours.

In [ ]:
place_details = None

if RUN_LIVE_REQUESTS and search_results and search_results.get("places"):
    selected = search_results["places"][0]
    place_id = selected["id"]
    place_details = places_json_request(
        f"{API_ROOT}/places/{quote(place_id, safe='')}",
        "id,displayName,formattedAddress,location,photos",
    )
    print("Place:", place_details.get("displayName", {}).get("text"))
    print("Address:", place_details.get("formattedAddress"))
    print("Photos listed:", len(place_details.get("photos", [])))
else:
    print("Offline mode, or no place selected. No details request sent.")

## 5 · Understand photographs and their authors

Photos are not supplied directly by the Text Search response. A Place Details response can provide a `photos` array, where each item has a resource name such as `places/PLACE_ID/photos/PHOTO_REFERENCE` and may include **author attributions**.

The **Place Photos (New)** endpoint requests the photo media by this resource name. A photo request may incur charges. Google requires relevant author attributions when photos are displayed, and there are restrictions on storing or redistributing Places content. Do not reproduce the old notebook's habit of downloading batches of photos into a permanent repository folder without verifying current terms.

The next cell displays attribution information, without fetching an image.

In [ ]:
photos = (place_details or {}).get("photos", [])

if photos:
    first_photo = photos[0]
    print("Photo resource available:", bool(first_photo.get("name")))
    for author in first_photo.get("authorAttributions", []):
        print("Author:", author.get("displayName", "(unnamed)"))
        print("Attribution link:", author.get("uri", "(none)"))
else:
    print("No live photo metadata loaded.")

### Optional: display one photo, without saving a file

The function below makes one **Place Photos (New)** request using a photo resource name and a maximum width of 640 pixels. The image is returned as bytes and displayed in the notebook **without writing it to disk**. The notebook itself may nevertheless retain displayed outputs if you save it, so **clear outputs before committing or sharing the notebook**. Display any required author attribution together with the image.

Because the request uses an API key, do not print the request headers or URLs containing credentials. We use the key in an HTTP header, not in a query string.

In [ ]:
def fetch_place_photo(photo_resource_name, max_width=640):
    """Return one Places API (New) photograph as image bytes."""
    api_key = os.getenv("GOOGLE_MAPS_API_KEY")
    if not api_key:
        raise RuntimeError("Set GOOGLE_MAPS_API_KEY first.")
    if not photo_resource_name.startswith("places/") or "/photos/" not in photo_resource_name:
        raise ValueError("Expected a Places API (New) photo resource name.")
    if not 1 <= max_width <= 4800:
        raise ValueError("max_width must be between 1 and 4800.")
    url = f"{API_ROOT}/{photo_resource_name}/media?" + urlencode(
        {"maxWidthPx": max_width}
    )
    req = Request(url, headers={"X-Goog-Api-Key": api_key})
    try:
        with urlopen(req, timeout=25) as response:
            if not response.headers.get("Content-Type", "").startswith("image/"):
                raise RuntimeError("Photo response was not an image.")
            return response.read()
    except HTTPError as exc:
        raise RuntimeError(f"Photo service returned HTTP {exc.code}.") from None

In [ ]:
if RUN_LIVE_REQUESTS and photos:
    from IPython.display import Image, display

    first_photo = photos[0]
    # Show the image in memory. Clear notebook outputs before sharing.
    display(Image(data=fetch_place_photo(first_photo["name"], max_width=640)))

    for author in first_photo.get("authorAttributions", []):
        print("Photo credit:", author.get("displayName", "(unnamed)"),
              "—", author.get("uri", ""))
else:
    print("No photograph requested. To test, enable live requests and load Place Details.")

## 6 · Urban design interpretation

Choose a place type that relates to an urban question, such as public libraries, public markets or community centres. Compare the three search results. What information can you obtain directly from the API? What is missing—for example, informal activity, accessibility, lived experience or the quality of public space?

For a short visual outcome, make a **small table** containing the name and coordinates of three places, accompanied by an annotated diagram showing the distinction between **Text Search → Place Details → Place Photo**. If you want to map the points, use an appropriate GIS tool and comply with Google's terms governing display, attribution and storage of Places data.

**Think critically:** A place-search API provides a selective commercial database. It does not offer a neutral description of urban life.

## Troubleshooting and safe practice

- **No results:** the query may be ambiguous, specific or not covered by the service. Try a clearer place description.
- **HTTP 400:** inspect the required field mask and valid request fields; don't expose the key when debugging.
- **HTTP 403:** check that Places API (New) is enabled, billing is active and key restrictions match your request.
- **HTTP 429:** check project quotas and usage limits; avoid sending repeated requests from notebook loops.
- **No photographs:** not every place has available photo metadata; try another result.
- **Notebook works offline but not online:** check your environment variable in the same running Jupyter process.

**Privacy and cost:** keep keys out of code and saved outputs; use only approved accounts; don't repeatedly fetch images; and review current terms for storage and photo attribution. Environment variables reduce accidental inclusion in source code but are not a complete secrets-management solution.

**Further reading:** [Google Text Search (New)](https://developers.google.com/maps/documentation/places/web-service/text-search), [Place Details (New)](https://developers.google.com/maps/documentation/places/web-service/place-details), [Place Photos (New)](https://developers.google.com/maps/documentation/places/web-service/place-photos), [API setup](API_SETUP.md).

**Verification:** No live billed request has been made during development of this notebook. Test the three live stages with an authorised Google Cloud project before describing them as fully tested.